In [ ]:
import json

with open("out_4o_4o-kind_4o-affirming_gitignore.json", "r") as f:
    data = json.load(f)["data"]

In [ ]:
from docent.data_models import AgentRun

runs: dict[str, list[AgentRun]] = {}

for config_run in data:
    config = config_run["config"]
    runs[config["system_prompt"]] = [AgentRun(**run) for run in config_run["results"]]

In [ ]:
system_prompts = [d["config"]["system_prompt"] for d in data]
print(system_prompts)
base_runs = runs[system_prompts[0]]
modified_runs = runs[system_prompts[2]]

In [ ]:
base_runs[0], modified_runs[0]

In [ ]:
from docent._ai_tools.diff import extract_states_and_diffs
from tqdm.auto import tqdm

all_diffs = []
pbar = tqdm(total=len(base_runs))

async def _run(base_run: AgentRun, modified_run: AgentRun):
    diffs = await extract_states_and_diffs(base_run, modified_run)
    all_diffs.append(diffs)
    pbar.update(1)


import anyio
async with anyio.create_task_group() as tg:
    for base_run, modified_run in list(zip(base_runs, modified_runs))[-20:]:
        tg.start_soon(_run, base_run, modified_run)

In [ ]:
all_diffs